# Ledger vs `algo.position`

On a sleeve-mismatch day: walk our running book against live `algo.position`
(outrights → consecutive 1m spreads) and surface where lots diverge.


In [ ]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = Path('/home/ajain/apeksha/pnl_attribution/src/v2')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import CONFIG, get_ch_client
from scripts.market_cache import _ch_dt64_utc
from scripts.overnight import BRENT_PRODUCT_ID, outrights_map_to_spreads
from scripts.position_timeline import (
    build_position_timeline,
    load_bod_spreads,
    map_to_series,
    position_asof,
)
from scripts.sleeve_attribution import attribute_range_vs_library
from scripts.transfers import naive_utc

client = get_ch_client()
BOOK = 'Hedger Spreads'
BOOK_ID = CONFIG.books[BOOK]
START = date(2026, 9, 17)
# pin a known mismatch day, or leave None to pick largest |diff| from range
DAY = date(2026, 10, 5)
# DAY = None

print(BOOK, BOOK_ID, '| pin', DAY)


### Pick day (optional)

If `DAY` is `None`, take the range day with the largest absolute sleeve vs library gross gap.


In [ ]:
if DAY is None:
    cmp = attribute_range_vs_library(client, BOOK, START)
    display(cmp.sort_values('diff', key=lambda s: s.abs(), ascending=False).head(10))
    DAY = cmp.loc[cmp['diff'].abs().idxmax(), 'date']
    if hasattr(DAY, 'date'):
        DAY = DAY.date()
    print('selected DAY =', DAY)
else:
    print('using pinned DAY =', DAY)


### Load ledger + algo.position snapshots

Our book starts from BOD `algo.position`, then applies same-day transfers/fills.
Compare at every `algo.position` timestamp on the London calendar day.


In [ ]:
bod_ts, bod = load_bod_spreads(client, DAY, BOOK)
tl = build_position_timeline(client, DAY, BOOK)
print('BOD curve', bod_ts, '| ledger events', len(tl))

algo = client.query_df(f"""
    SELECT timestamp, positions
    FROM algo.position
    WHERE book_id = {BOOK_ID}
      AND product_id = {BRENT_PRODUCT_ID}
      AND toDate(toTimeZone(timestamp, 'Europe/London')) = toDate('{DAY}')
    ORDER BY timestamp
""")
algo['timestamp'] = algo['timestamp'].map(naive_utc)
print('algo.position snaps', len(algo),
      '| first', algo['timestamp'].iloc[0] if len(algo) else None,
      '| last', algo['timestamp'].iloc[-1] if len(algo) else None)


### Diff at each algo.position timestamp

`ours` = ledger after last event ≤ snap time. `algo` = outrights→spreads at that snap.
`gap_lots` = Σ |ours − algo| across tenors.


In [ ]:
rows = []
for ts, pos_map in algo[['timestamp', 'positions']].itertuples(index=False, name=None):
    ours = map_to_series(position_asof(tl, ts))
    thrs = outrights_map_to_spreads(pos_map)
    gap = ours.subtract(thrs, fill_value=0.0)
    gap = gap[gap.abs() > 1e-9]
    rows.append({
        'timestamp': ts,
        'ours_lots': float(ours.abs().sum()),
        'algo_lots': float(thrs.abs().sum()),
        'gap_lots': float(gap.abs().sum()),
        'signed_gap': float(gap.sum()),
        'n_tenor_diff': int(len(gap)),
    })

diff = pd.DataFrame(rows)
print('snaps with gap > 0:', int((diff['gap_lots'] > 1e-9).sum()), '/', len(diff))
print('max gap_lots', round(float(diff['gap_lots'].max()), 4) if len(diff) else None)
first_break = diff.loc[diff['gap_lots'] > 1e-9, 'timestamp']
print('first break', first_break.iloc[0] if len(first_break) else None)
display(diff.loc[diff['gap_lots'] > 1e-9].head(20))
display(diff.tail(5))


### Tenor detail at worst gap / EOD

Also list ledger events around the first break.


In [ ]:
def tenor_gap_at(ts) -> pd.DataFrame:
    row = algo.loc[algo['timestamp'] == ts].iloc[0]
    ours = map_to_series(position_asof(tl, ts))
    thrs = outrights_map_to_spreads(row['positions'])
    out = pd.DataFrame({'ours': ours, 'algo': thrs}).fillna(0.0)
    out['gap'] = out['ours'] - out['algo']
    return out.loc[out['gap'].abs() > 1e-9]

if not diff.empty and float(diff['gap_lots'].max()) > 1e-9:
    worst_ts = diff.loc[diff['gap_lots'].idxmax(), 'timestamp']
    eod_ts = diff['timestamp'].iloc[-1]
    print('worst snap', worst_ts, '| gap_lots', round(float(diff['gap_lots'].max()), 4))
    display(tenor_gap_at(worst_ts))
    print('EOD snap', eod_ts, '| gap_lots',
          round(float(diff.loc[diff['timestamp'] == eod_ts, 'gap_lots'].iloc[0]), 4))
    display(tenor_gap_at(eod_ts))

    t0 = first_break.iloc[0]
    win = tl.loc[
        (tl['timestamp'] >= t0 - pd.Timedelta(minutes=5))
        & (tl['timestamp'] <= t0 + pd.Timedelta(minutes=5)),
        ['timestamp', 'reason', 'id', 'cycle_id', 'outbound', 'source_book',
         'position_before', 'position_after'],
    ]
    print('ledger events ±5m around first break')
    display(win)
else:
    print('no lot gaps on this day')
